In [33]:
import os
import json
from langchain_community.graphs import Neo4jGraph
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document
from langchain_neo4j import Neo4jVector
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_classic.chains import RetrievalQA
from langchain.chat_models import init_chat_model
import textwrap
load_dotenv()

True

In [34]:
NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE') or 'neo4j'
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

VECTOR_INDEX_NAME = 'brd_chunks'
VECTOR_NODE_LABEL = 'Chunk'
VECTOR_SOURCE_PROPERTY = 'text'
VECTOR_EMBEDDING_PROPERTY = 'textEmbedding'

In [35]:
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
GOOGLE_ENDPOINT_URL = "https://generativelanguage.googleapis.com/v1beta2/models/text-embedding-3-small:embedContent"
# "https://ai.google.dev/api/embeddings#method:-models.embedcontent"

In [4]:
print(NEO4J_DATABASE)

84afaea0


In [5]:
brd_file_name = "C:\\Users\\HP\\OneDrive\\Desktop\\Dinesh\\TCS\\Apple\\Tasks\\projects\\graphrag\\parsed_brd.json"
brd_file_as_object = json.load(open(brd_file_name))
type(brd_file_as_object)

dict

In [6]:
for k,v in brd_file_as_object.items():
    print(k, type(v))

Features <class 'str'>
Modules <class 'str'>
UserStories <class 'str'>
Issues <class 'str'>


In [7]:
feature_text = brd_file_as_object['Features']

In [8]:
feature_text

'This BRD outlines the business objectives, stakeholders, functional and non-functional requirements, scope, constraints, and success criteria for React ChatBotify.\n2. Business Objectives\n2.1 Primary Objectives\nProvide a reusable chatbot component for React applications.\nEnable developers to easily configure chatbot conversations.\nSupport enterprise-level customization (themes, plugins, settings, styles).\nMaintain backward compatibility across React versions (16–19).\nEnsure strong extensibility via hooks and event-driven architecture.\n2.2 Secondary Objectives\nEnable analytics integration.\nSupport AI-driven chatbot logic.\nAllow plugin ecosystem growth.\nSupport mobile-first responsive behavior.\nImprove developer experience (DX).\n3. Project Scope\n3.1 In Scope\nReact chatbot component library\nConversation flow management\nSettings and style customization\nEvent-driven architecture\nPlugin framework\nTheme support\nAudio & voice services\nMobile support optimization\nUnit, i

In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap  = 100,
    length_function = len,
    is_separator_regex = False,
)


In [10]:
feature_text_chunks = text_splitter.split_text(feature_text)

In [11]:
type(feature_text_chunks)

list

In [12]:
len(feature_text_chunks)

5

In [13]:
feature_text_chunks[0]

'This BRD outlines the business objectives, stakeholders, functional and non-functional requirements, scope, constraints, and success criteria for React ChatBotify.\n2. Business Objectives\n2.1 Primary Objectives\nProvide a reusable chatbot component for React applications.\nEnable developers to easily configure chatbot conversations.\nSupport enterprise-level customization (themes, plugins, settings, styles).\nMaintain backward compatibility across React versions (16–19).\nEnsure strong extensibility via hooks and event-driven architecture.\n2.2 Secondary Objectives\nEnable analytics integration.\nSupport AI-driven chatbot logic.\nAllow plugin ecosystem growth.\nSupport mobile-first responsive behavior.\nImprove developer experience (DX).\n3. Project Scope\n3.1 In Scope\nReact chatbot component library\nConversation flow management\nSettings and style customization\nEvent-driven architecture\nPlugin framework\nTheme support\nAudio & voice services\nMobile support optimization'

In [14]:
def split_brd_data_from_file(file):
    chunks_with_metadata = [] # use this to accumlate chunk records
    file_as_object = json.load(open(file)) # open the json file
    for item in ['Features','Modules','UserStories','Issues']: # pull these keys from the json
        print(f'Processing {item} from {file}') 
        item_text = file_as_object[item] # grab the text of the item
        item_text_chunks = text_splitter.split_text(item_text) # split the text into chunks
        chunk_seq_id = 0
        for chunk in item_text_chunks[:10]: # only take the first 10 chunks
            form_id = 'brd_document' # extract form id from file name
            # finally, construct a record with metadata and the chunk text
            chunks_with_metadata.append({
                'text': chunk, 
                # metadata from looping...
                'brdItem': item,
                'chunkSeqId': chunk_seq_id,
                # constructed metadata...
                'formId': f'{form_id}', # pulled from the filename
                'chunkId': f'{form_id}-{item}-chunk{chunk_seq_id:04d}',
                # metadata from file...
                # 'names': file_as_object['names'],
                # 'cik': file_as_object['cik'],
                # 'cusip6': file_as_object['cusip6'],
                # 'source': file_as_object['source'],
            })
            chunk_seq_id += 1
        print(f'\tSplit into {chunk_seq_id} chunks')
    return chunks_with_metadata

In [23]:
brd_file_chunks = split_brd_data_from_file(brd_file_name)

Processing Features from C:\Users\HP\OneDrive\Desktop\Dinesh\TCS\Apple\Tasks\projects\graphrag\parsed_brd.json
	Split into 5 chunks
Processing Modules from C:\Users\HP\OneDrive\Desktop\Dinesh\TCS\Apple\Tasks\projects\graphrag\parsed_brd.json
	Split into 1 chunks
Processing UserStories from C:\Users\HP\OneDrive\Desktop\Dinesh\TCS\Apple\Tasks\projects\graphrag\parsed_brd.json
	Split into 3 chunks
Processing Issues from C:\Users\HP\OneDrive\Desktop\Dinesh\TCS\Apple\Tasks\projects\graphrag\parsed_brd.json
	Split into 1 chunks


In [16]:
kg = Neo4jGraph(
    url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD, database=NEO4J_DATABASE
)

C:\Users\HP\AppData\Local\Temp\ipykernel_27988\336443517.py:1: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``.
  kg = Neo4jGraph(


In [18]:
kg.query("""
CREATE CONSTRAINT unique_chunk IF NOT EXISTS 
    FOR (c:Chunk) REQUIRE c.chunkId IS UNIQUE
""")


[]

In [19]:
kg.query("SHOW INDEXES")

[{'id': 1,
  'name': 'index_1b9dcc97',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'LOOKUP',
  'entityType': 'RELATIONSHIP',
  'labelsOrTypes': None,
  'properties': None,
  'indexProvider': 'token-lookup-1.0',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': 0},
 {'id': 0,
  'name': 'index_460996c0',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'LOOKUP',
  'entityType': 'NODE',
  'labelsOrTypes': None,
  'properties': None,
  'indexProvider': 'token-lookup-1.0',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': 0},
 {'id': 2,
  'name': 'unique_chunk',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'RANGE',
  'entityType': 'NODE',
  'labelsOrTypes': ['Chunk'],
  'properties': ['chunkId'],
  'indexProvider': 'range-1.0',
  'owningConstraint': 'unique_chunk',
  'lastRead': None,
  'readCount': 0}]

In [20]:
merge_chunk_node_query = """
MERGE(mergedChunk:Chunk {chunkId: $chunkParam.chunkId})
    ON CREATE SET 
        mergedChunk.brdItem = $chunkParam.brdItem, 
        mergedChunk.chunkSeqId = $chunkParam.chunkSeqId, 
        mergedChunk.text = $chunkParam.text
RETURN mergedChunk
"""

In [ ]:
node_count = 0
for chunk in brd_file_chunks:
    print(f"Creating `:Chunk` node for chunk ID {chunk['chunkId']}")
    kg.query(merge_chunk_node_query, 
            params={
                'chunkParam': chunk
            })
    node_count += 1
print(f"Created {node_count} nodes")

Creating `:Chunk` node for chunk ID brd_document-Features-chunk0000
Creating `:Chunk` node for chunk ID brd_document-Features-chunk0001
Creating `:Chunk` node for chunk ID brd_document-Features-chunk0002
Creating `:Chunk` node for chunk ID brd_document-Features-chunk0003
Creating `:Chunk` node for chunk ID brd_document-Features-chunk0004
Creating `:Chunk` node for chunk ID brd_document-Modules-chunk0000
Creating `:Chunk` node for chunk ID brd_document-UserStories-chunk0000
Creating `:Chunk` node for chunk ID brd_document-UserStories-chunk0001
Creating `:Chunk` node for chunk ID brd_document-UserStories-chunk0002
Creating `:Chunk` node for chunk ID brd_document-Issues-chunk0000
Created 10 nodes


In [22]:
kg.query("""
         MATCH (n)
         RETURN count(n) as nodeCount
         """)

[{'nodeCount': 10}]

In [26]:
kg.query("""
         CREATE VECTOR INDEX `brd_chunks` IF NOT EXISTS
          FOR (c:Chunk) ON (c.textEmbedding) 
          OPTIONS { indexConfig: {
            `vector.dimensions`: 1536,
            `vector.similarity_function`: 'cosine'    
         }}
""")

[]

In [28]:
kg.query("SHOW INDEXES")

[{'id': 4,
  'name': 'brd_chunks',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'VECTOR',
  'entityType': 'NODE',
  'labelsOrTypes': ['Chunk'],
  'properties': ['textEmbedding'],
  'indexProvider': 'vector-3.0',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': 0},
 {'id': 1,
  'name': 'index_1b9dcc97',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'LOOKUP',
  'entityType': 'RELATIONSHIP',
  'labelsOrTypes': None,
  'properties': None,
  'indexProvider': 'token-lookup-1.0',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': 0},
 {'id': 0,
  'name': 'index_460996c0',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'LOOKUP',
  'entityType': 'NODE',
  'labelsOrTypes': None,
  'properties': None,
  'indexProvider': 'token-lookup-1.0',
  'owningConstraint': None,
  'lastRead': neo4j.time.DateTime(2026, 3, 5, 9, 21, 4, 807000000, tzinfo=<UTC>),
  'readCount': 19},
 {'id': 2,
  'name': 'unique_chunk',
  'state': 'ONLINE',
  

In [6]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001",output_dimensionality=1536)

In [ ]:
existing_graph = Neo4jVector.from_existing_graph(
    # embedding=GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001",output_dimensionality=1536),
    embedding=embeddings,
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    index_name=VECTOR_INDEX_NAME,
    node_label=VECTOR_NODE_LABEL,
    text_node_properties=[VECTOR_SOURCE_PROPERTY],
    embedding_node_property=VECTOR_EMBEDDING_PROPERTY,
    
)

In [33]:
# kg.query("""
#     MATCH (chunk:Chunk) WHERE chunk.textEmbedding IS NULL
#     WITH chunk, genai.vector.encode(
#       chunk.text, 
#       "OpenAI", 
#       {
#         token: $googleApiKey, 
#         endpoint: $googleEndpoint
#       }) AS vector
#     CALL db.create.setNodeVectorProperty(chunk, "textEmbedding", vector)
#     """, 
#     params={"googleApiKey":GOOGLE_API_KEY, "googleEndpoint": GOOGLE_ENDPOINT_URL} )

In [32]:
kg.refresh_schema()
print(kg.schema)


Node properties:
Chunk {chunkId: STRING, brdItem: STRING, chunkSeqId: INTEGER, text: STRING, textEmbedding: LIST}
Relationship properties:

The relationships:



In [35]:
result = existing_graph.similarity_search("what are the features of the product?", k=1)

In [36]:
result[0].page_content


'\ntext: This BRD outlines the business objectives, stakeholders, functional and non-functional requirements, scope, constraints, and success criteria for React ChatBotify.\n2. Business Objectives\n2.1 Primary Objectives\nProvide a reusable chatbot component for React applications.\nEnable developers to easily configure chatbot conversations.\nSupport enterprise-level customization (themes, plugins, settings, styles).\nMaintain backward compatibility across React versions (16–19).\nEnsure strong extensibility via hooks and event-driven architecture.\n2.2 Secondary Objectives\nEnable analytics integration.\nSupport AI-driven chatbot logic.\nAllow plugin ecosystem growth.\nSupport mobile-first responsive behavior.\nImprove developer experience (DX).\n3. Project Scope\n3.1 In Scope\nReact chatbot component library\nConversation flow management\nSettings and style customization\nEvent-driven architecture\nPlugin framework\nTheme support\nAudio & voice services\nMobile support optimization'

In [45]:
# cypher = """MATCH (chunk:Chunk)
# SET chunk.formId = 'brd_document'
# RETURN chunk;
# """
# kg.query(cypher)

In [43]:
cypher = """
  MATCH (from_same_section:Chunk)
  WHERE from_same_section.formId = $formIdParam
    AND from_same_section.brdItem = $brdItemParam
  WITH from_same_section
    ORDER BY from_same_section.chunkSeqId ASC
  WITH collect(from_same_section) as section_chunk_list
    CALL apoc.nodes.link(
        section_chunk_list, 
        "NEXT", 
        {avoidDuplicates: true}
    )
  RETURN size(section_chunk_list)
"""
for brdItemName in ['Features','Modules','UserStories','Issues']:
  kg.query(cypher, params={'formIdParam':'brd_document', 
                           'brdItemParam': brdItemName})


In [44]:
kg.refresh_schema()
print(kg.schema)


Node properties:
Chunk {chunkId: STRING, brdItem: STRING, chunkSeqId: INTEGER, text: STRING, textEmbedding: LIST, formId: STRING}
Relationship properties:

The relationships:
(:Chunk)-[:NEXT]->(:Chunk)


In [46]:
cypher = """
  MATCH (anyChunk:Chunk) 
  WITH anyChunk LIMIT 1
  RETURN anyChunk {.formId } as BRDInfo
"""
form_info_list = kg.query(cypher)

form_info_list


[#FDEE]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('p-mt-3b814dd7cab7-1-0014.production-orch-0068.neo4j.io', 7687)) (ResolvedIPv6Address(('64:ff9b::227e:ab19', 7687, 0, 0))): OSError('No data')
Transaction failed and will be retried in 1.1901983404287113s (Failed to read from defunct connection IPv4Address(('p-mt-3b814dd7cab7-1-0014.production-orch-0068.neo4j.io', 7687)) (ResolvedIPv6Address(('64:ff9b::227e:ab19', 7687, 0, 0))))


[{'BRDInfo': {'formId': 'brd_document'}}]

In [48]:
brd_info = form_info_list[0]['BRDInfo']

In [49]:
cypher = """
    MERGE (f:BRDInfo {formId: $BRDInfoParam.formId })
      ON CREATE 
        SET f.names = 'BRD Document'
"""

kg.query(cypher, params={'BRDInfoParam': brd_info})

[]

In [50]:
cypher = """
  MATCH (c:Chunk), (f:BRDInfo)
    WHERE c.formId = f.formId
  MERGE (c)-[newRelationship:PART_OF]->(f)
  RETURN count(newRelationship)
"""

kg.query(cypher)

[{'count(newRelationship)': 10}]

In [51]:
cypher = """
  MATCH (first:Chunk), (f:BRDInfo)
  WHERE first.formId = f.formId
    AND first.chunkSeqId = 0
  WITH first, f
    MERGE (f)-[r:SECTION {BRDItem: first.brdItem}]->(first)
  RETURN count(r)
"""

kg.query(cypher)

[{'count(r)': 4}]

In [36]:
retrieval_query_window = """
MATCH window=
    (:Chunk)-[:NEXT*0..1]->(node)-[:NEXT*0..1]->(:Chunk)
WITH node, score, window as longestWindow 
  ORDER BY length(window) DESC LIMIT 1
WITH nodes(longestWindow) as chunkList, node, score
  UNWIND chunkList as chunkRows
WITH collect(chunkRows.text) as textList, node, score
RETURN apoc.text.join(textList, " \n ") as text,
    score,
    node {.source} AS metadata
"""
vector_store_window = Neo4jVector.from_existing_index(
    embedding=embeddings,
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
    index_name=VECTOR_INDEX_NAME,
    text_node_property=VECTOR_SOURCE_PROPERTY,
    retrieval_query=retrieval_query_window, # NEW!!!
)

# Create a retriever from the vector store
retriever_window = vector_store_window.as_retriever()


In [37]:

# Create a chatbot Question & Answer chain from the retriever
chain_window = RetrievalQA.from_chain_type(
    # ChatOpenAI(temperature=0), 
    ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0),
    chain_type="stuff", 
    retriever=retriever_window
)

In [28]:
question = "What are the features of the product?"
answer = chain_window(
    {"query": question},
    return_only_outputs=True,
)
print(textwrap.fill(answer['result']))


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `source` does not exist in database `84afaea0`. Verify that the spelling is correct.', position=<SummaryInputPosition line=12, column=12, offset=511>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 511, 'line': 12, 'column': 12}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k \nMATCH window=\n    (:Chunk)-[:NEXT*0..1]->(node)-[:NEXT*0..1]->(:Chunk)\nWITH node, score, window as longestWindow \n  ORDER BY length(window) DESC LIMIT 1\nW

Based on the provided requirements, the product features are:  **Core
Chatbot Functionality:** *   Renders a chatbot UI component. *
Manages user inputs and bot responses. *   Supports audio playback of
messages. *   Supports voice input via microphone. *   Allows audio
and voice features to be toggled.  **Conversation Flow & Logic:** *
Allows configurable conversation flows. *   Supports a block-based
conversation structure. *   Enables developers to define conversation
blocks using JSON/TS objects. *   Each block supports attributes like
messages, options, and validation. *   Supports conversation paths
logic. *   Supports conditional navigation between blocks. *
Supports custom input validation logic.  **Customization & Styling:**
*   Accepts a `settings` prop for functional configurations. *
Accepts a `styles` prop for UI customization. *   Loads default
settings if no configuration is supplied, and safely merges user
settings. *   Supports theme-based styling. *   Allows themes to

In [29]:
question = "What is the high level overview of the product architecture layer?"
answer = chain_window(
    {"query": question},
    return_only_outputs=True,
)
print(textwrap.fill(answer['result']))


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `source` does not exist in database `84afaea0`. Verify that the spelling is correct.', position=<SummaryInputPosition line=12, column=12, offset=511>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 511, 'line': 12, 'column': 12}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k \nMATCH window=\n    (:Chunk)-[:NEXT*0..1]->(node)-[:NEXT*0..1]->(:Chunk)\nWITH node, score, window as longestWindow \n  ORDER BY length(window) DESC LIMIT 1\nW

React ChatBotify has evolved into a feature-rich v2 architecture with
the following high-level components and design principles:  *
**Modular folder structure:** Indicating a well-organized and scalable
codebase. *   **Separation of Settings and Styles:** Clearly
distinguishing functional configurations from UI customization. *
**Custom Events system:** An event-driven architecture for dispatching
and subscribing to lifecycle events. *   **Internal and External
Hooks:** Providing extensibility and integration points. *   **Themes
repository:** Supporting a modular approach to UI theming. *
**Experimental Plugin architecture:** Enabling further extensibility
and ecosystem growth.


In [31]:
question = "What are the constraints?"
answer = chain_window(
    {"query": question},
    return_only_outputs=True,
)
print(textwrap.fill(answer['result']))


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `source` does not exist in database `84afaea0`. Verify that the spelling is correct.', position=<SummaryInputPosition line=12, column=12, offset=511>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 511, 'line': 12, 'column': 12}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k \nMATCH window=\n    (:Chunk)-[:NEXT*0..1]->(node)-[:NEXT*0..1]->(:Chunk)\nWITH node, score, window as longestWindow \n  ORDER BY length(window) DESC LIMIT 1\nW

Based on the provided functional requirements, here are the
constraints:  *   **Technology/Tooling Constraints:**     *   Unit
testing must be done via Jest (FR32).     *   Integration testing must
be done via Cypress (FR33).     *   Compatibility testing is required
across React versions 16–19 (FR34).     *   CI/CD must validate builds
before release (FR35). *   **Design/Architectural Constraints:**     *
Conversation blocks must be defined via JSON/TS objects (FR6).     *
Internal hooks must remain encapsulated (FR18).     *   Plugins must
remain loosely coupled (FR28).     *   Themes must be externally
maintainable (FR24).     *   User settings must merge safely with
default settings (FR13).     *   External hooks must act as a
filtering layer over internal logic (FR19). *   **Development/API
Constraints:**     *   Developers must define conversation blocks via
JSON/TS objects (FR6).     *   Developers must subscribe to events
externally (FR15).
